# 02 — Compute limits: AVX-512 and AMX

**Question.** What sustained arithmetic rate is attainable, how does it scale with cores, and how much of the AMX instruction capability does a library GEMM realize?

FP64 and FP32 use independent register-resident AVX-512 FMA chains. BF16 is measured in two distinct ways: native AMX tile instructions and a warmed PyTorch/oneDNN GEMM. Multiply and add count as two FLOPs. These are different workloads; their ratios are not application speedups.


In [ ]:
from pathlib import Path
import hashlib, json, sys
import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench.analysis import (load_measurements, select_measurements, export_figure,
    memory_table, compute_table, locality_table, scaling_table, knees_table,
    theoretical_table, concurrent_table, bandwidth_scaling, cache_working_set,
    locality_figure, compute_scaling, fp64_rooflines, dual_scaling)
pio.renderers.default = 'notebook'
validation = json.loads((ROOT / 'data/processed/validation.json').read_text())
assert validation['status'] == 'passed', validation['errors']
assert validation['dataset_sha256'] == hashlib.sha256((ROOT / 'data/processed/measurements.csv').read_bytes()).hexdigest()
assert validation['selection_sha256'] == hashlib.sha256((ROOT / 'configs/analysis.json').read_bytes()).hexdigest()
df = select_measurements(load_measurements(ROOT / 'data/processed/measurements.csv'))
NOTEBOOK = '02_compute_limits'
def table(frame):
    shown = frame.copy()
    if 'implementation' in shown:
        shown['implementation'] = shown['implementation'].replace({
            'native_amx_4_tiles': 'Native AMX tiles',
            'onednn_numactl_only_omp_affinity_unset': 'oneDNN GEMM',
            'native_avx512': 'AVX-512'})
    if 'metric' in shown:
        shown['metric'] = shown['metric'].replace({'bandwidth_gbs':'GB/s','performance_gops':'GFLOP/s','latency_ns':'ns/load'})
    shown = shown.rename(columns={'memory_mode':'Mode','socket':'Socket','memory_target':'Memory',
        'locality':'Placement','working_set_gib':'Working set (GiB)','implementation':'Compute path',
        'dtype':'Precision','m':'Matrix size','n_samples':'Samples','efficiency':'Scaling efficiency'})
    display(shown.round(3).style.set_uuid(NOTEBOOK).hide(axis='index'))


## 1. AVX-512 FP64 and FP32 scaling

**Conditions.** Twelve independent vector accumulators, local CPU binding, five duration-based samples at each physical-core count. The illustrated curves use the Flat node; the numerical table below retains both measured modes and sockets.

In [ ]:
fig = compute_scaling(df, 'avx512')
export_figure(fig, 'avx512_scaling', NOTEBOOK)

In [ ]:
c = compute_table(df)
a = c[c.dtype.isin(['fp64','fp32'])].copy()
a['median_TFLOP_s'] = a['median']/1000; a['std_TFLOP_s'] = a['std']/1000
table(a[['memory_mode','socket','dtype','median_TFLOP_s','std_TFLOP_s','n_samples']])

**Implication.** These rates give a measured compute reference for register-resident arithmetic. Memory traffic, dependencies, and scheduling can lower application throughput. FP32 and FP64 remain distinct arithmetic paths.

## 2. Native AMX capability and achieved BF16 GEMM

**Conditions.** The native kernel uses four independent destination tiles and verifies their expected numerical values. GEMM uses 6144³ BF16 matrices, three warm-ups, explicit CPU/NUMA binding, and no inner OpenMP placement override. AMX execution is checked during an untimed oneDNN warm-up. Five timed windows, each scheduled for at least one second, amortize startup overhead.

In [ ]:
fig = compute_scaling(df, 'amx')
export_figure(fig, 'bf16_scaling', NOTEBOOK)

In [ ]:
b = c[c.dtype.eq('bf16') & (c.m.isna() | c.m.eq(6144))].copy()
b['median_TFLOP_s'] = b['median']/1000; b['std_TFLOP_s'] = b['std']/1000
table(b[['memory_mode','socket','implementation','m','median_TFLOP_s','std_TFLOP_s','n_samples']])
for mode in ['flat','cache']:
    for socket in [0,1]:
        part=b[b.memory_mode.eq(mode)&b.socket.eq(socket)].set_index('implementation')
        native=part.loc['native_amx_4_tiles','median_TFLOP_s']; gemm=part.loc['onednn_numactl_only_omp_affinity_unset','median_TFLOP_s']
        display(Markdown(f"**Result, {mode}, socket {socket}.** Native tiles sustain {native:.2f} TFLOP/s; the 6144³ GEMM achieves {gemm:.2f} TFLOP/s ({gemm/native:.1%} of the native instruction rate)."))

**Implication.** Native tile throughput characterizes an instruction microbenchmark. Achieved GEMM includes data movement, packing, library scheduling, and output conversion. The measured gap is headroom relative to this microbenchmark, not a promise that an application can recover it.

### Does GEMM matrix size change the result?

The same full-core measurement is repeated at 4096³, 6144³, and 8192³; rows remain separate by shape and socket.

In [ ]:
shapes=c[c.implementation.eq('onednn_numactl_only_omp_affinity_unset')].copy()
shapes['median_TFLOP_s']=shapes['median']/1000
shapes['std_TFLOP_s']=shapes['std']/1000
table(shapes[['memory_mode','socket','m','median_TFLOP_s','std_TFLOP_s','n_samples']])

## 3. Architectural reference and attainable rates

The clock model uses 56 cores and explicit 1.90 GHz base and 3.50 GHz maximum-turbo reference clocks. Maximum turbo is not an all-core AVX-512/AMX guarantee. The table compares rates without calling the nominal-clock ratio hardware utilization.

Sources: [Intel Xeon CPU Max overview](https://www.intel.com/content/www/us/en/developer/articles/technical/xeon-scalable-processor-max-series.html), [Max 9480 specifications](https://www.intel.com/content/www/us/en/products/sku/232592/intel-xeon-cpu-max-9480-processor-112-5m-cache-1-90-ghz/specifications.html), and [operations-per-cycle reference](https://download.intel.com/newsroom/2021/client-computing/intel-architecture-day-2021-presentation.pdf).

In [ ]:
t=theoretical_table(df)
table(t[t.memory_mode.eq('flat') & t.socket.eq(0) & (t.m.isna() | t.m.eq(6144))])

**Limits of interpretation.** This study does not infer throttling, energy efficiency, or sustained ISA frequency from the achieved rate. Frequency snapshots are metadata rather than a measurement of active-core frequency throughout the timed window. Continue with Rooflines and full-node scaling.